In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
dir1 = os.path.dirname(os.path.abspath('..'))
if not dir1 in sys.path: sys.path.append(dir1)

In [2]:
from pathlib import Path

import pickle
import numpy as np

from utils.plots import *
from utils import models
from jax import random
from jax import numpy as jnp

from utils.infer import bayesian_pd_mcmc
from utils.infer import run_bayesian_pd

In [3]:
param_dir = Path('../../prior_params')
bpd_path = Path('rarker_bpd.pkl')
out_dir = Path('post_samples')
out_dir.mkdir(parents=True, exist_ok=True)

prior = dict(
    alpha_hr  = np.load(param_dir / "alpha_hr.npy"),
    lambd_hr  = np.load(param_dir / "lambd_hr.npy"),
    alpha_zpp = np.load(param_dir / "alpha_zpp.npy"),
    mu_w      = np.load(param_dir / "mu_w.npy"),
    L_w   = np.load(param_dir / "L_w.npy"),
    f_h       = np.load(param_dir / "f_h.npy"),
    g       = np.load(param_dir / "g.npy"),
)

with open(bpd_path, 'rb') as f:
    bpd = pickle.load(f)

z = bpd['z']
w = bpd['w']
f_h = bpd['f_h']
g = bpd['g']
a = bpd['a']
b = bpd['b']
f_b = bpd['f_b']


In [4]:
duration = 3
fs = 64
snr = 10

num_warmup = 1000
num_samples = 10000

experiments = [
    {'dur': 5, 'fs': 64, 'snr':10},       
    {'dur': 5, 'fs': 16, 'snr':10},
    {'dur': 5, 'fs': 64, 'snr':1},
    {'dur': 5, 'fs': 16, 'snr':1},
]

tagify = lambda exp : f'dur_{exp["dur"]:0g}_fs_{exp["fs"]:0g}_snr_{exp["snr"]:0g}'


for exp in experiments:
    dur = exp['dur']
    fs = exp['fs']
    snr = exp['snr']
    t = np.arange(0, dur, 1/fs)

    _, y_true = models.evaluate(t, z, w, f_h, g, a, b, f_b)
    y_true_power = jnp.mean((y_true-y_true.mean())**2)
    sigma_e = jnp.sqrt(y_true_power/snr)
    rkey = random.key(0)
    r = random.normal(rkey, t.shape) * sigma_e
    y_obs = y_true + r

    outpath = out_dir / f'{tagify(exp)}_map.pkl'
    if outpath.exists():
        with open(outpath, 'rb') as f:
            soln_map = pickle.load(f)
    else:
        soln_map = run_bayesian_pd(t, y_obs, prior, seed=0)
        with open(outpath, 'wb') as f:
            pickle.dump(soln_map, f)

    obs = dict(
        t = t,
        y = y_obs,
    )
    config = dict(
        t0 = soln_map['t0'],
        f_b = soln_map['f_b'],
        M = soln_map['M'],
    )
    init = dict(
        a = soln_map['a'],
        b = soln_map['b'],
        sigma_e = soln_map['sigma_e'],
    )

    init_vars = ['hr', 'zpp', 'zm*', 'w*']
    init_vals = {k:soln_map[k] for k in init_vars}

    outpath = out_dir / f'{tagify(exp)}_prior_obs_init_config.pkl'
    if not outpath.exists():
        with open(outpath, 'wb') as f:
            pickle.dump((prior, obs, init, config), f)

    outpath = out_dir / f'{tagify(exp)}_post_samples.pkl'
    if outpath.exists():
        with open(outpath, 'rb') as f:
            post_samples = pickle.load(f)
    else:
        post_samples = bayesian_pd_mcmc(prior, obs, config, init, init_vals,
                                        num_samples=num_samples, num_warmup=num_warmup)
        with open(outpath, 'wb') as f:
            pickle.dump(post_samples, f)
